# CloudScale AI — Phase 1 Training (Google Colab)

Trains the PPO Reinforcement Learning agent on the K8s+Spot Gymnasium simulator and streams every experiment to your **Weights & Biases** dashboard.

**Before you run:** set your runtime to **GPU → T4** (`Runtime` → `Change runtime type`).

## 1. Set your W&B API key

Get it from <https://wandb.ai/authorize>. Use Colab Secrets (🔑 icon in the left panel).

| Secret name | Value |
|---|---|
| `WANDB_API_KEY` | Your W&B API key |

Training will stop immediately if this key is missing or invalid.

In [ ]:
from google.colab import userdata
import os

wandb_key = ''
try:
    wandb_key = userdata.get('WANDB_API_KEY') or ''
    wandb_key = wandb_key.strip()
except Exception:
    pass

if wandb_key:
    os.environ['WANDB_API_KEY'] = wandb_key
    print(f'\u2705 W&B API key loaded (length: {len(wandb_key)})')
else:
    print('\u274c WANDB_API_KEY not found \u2014 add it to Colab Secrets (\ud83d\udd11 icon)')

## 2. Clone your repo + install deps

**Important:** do not reinstall or downgrade Colab's core compiled stack (`torch`, `numpy`, `protobuf`, etc.). Colab pre-installs compiled packages together; changing them inside the live runtime can cause `numpy.dtype size changed` binary-compatibility errors.

In [ ]:
%cd /content
!rm -rf CloudScale-AI-Autonomous-FinOps-Orchestrator
!git clone https://github.com/bhanujjj/CloudScale-AI-Autonomous-FinOps-Orchestrator.git
%cd CloudScale-AI-Autonomous-FinOps-Orchestrator

from importlib import metadata
keep_packages = ['numpy', 'protobuf', 'httpx', 'rich', 'tenacity', 'pydantic', 'typing_extensions']
with open('/tmp/colab-keep-versions.txt', 'w') as f:
    for package in keep_packages:
        try:
            version = metadata.version(package)
        except metadata.PackageNotFoundError:
            continue
        f.write(f'{package}=={version}\n')

%pip install -q -r cloudscale/configs/requirements.txt -c /tmp/colab-keep-versions.txt --upgrade-strategy only-if-needed

import gymnasium, numpy, stable_baselines3, torch, wandb
print('Python executable ready')
print('numpy:', numpy.__version__)
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
print('gymnasium:', gymnasium.__version__)
print('stable-baselines3:', stable_baselines3.__version__)
print('wandb:', wandb.__version__)

TRAIN_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('training device:', TRAIN_DEVICE)

## 3. Sanity-check the env (5 random steps)

In [ ]:
from cloudscale.envs.k8s_spot_env import K8sSpotEnv

env = K8sSpotEnv(render_mode='ansi')
obs, info = env.reset(seed=0)
print('obs shape:', obs.shape, '| action space:', env.action_space.n)

for i in range(5):
    a = env.action_space.sample()
    obs, r, term, trunc, info = env.step(a)
    print(f'  step {i}: action={a} reward={r:.4f} terminated={term} truncated={trunc}')

print('env smoke test: PASS')

## 4. Single training run (200k timesteps, ~5 min on T4)

W&B must connect before training starts. If the API key is missing, this cell exits immediately.

In [ ]:
%cd /content/CloudScale-AI-Autonomous-FinOps-Orchestrator
!python -m cloudscale.training.train_ppo \
    --mode single \
    --total-timesteps 200000 \
    --eval-episodes 5 \
    --device $TRAIN_DEVICE

## 5. (Optional) Optuna sweep — 20 trials, ~20-30 min on T4

In [ ]:
%cd /content/CloudScale-AI-Autonomous-FinOps-Orchestrator
!python -m cloudscale.training.train_ppo \
    --mode optuna \
    --n-trials 20 \
    --total-timesteps 100000 \
    --device $TRAIN_DEVICE

## 6. Open your W&B dashboard

→ <https://wandb.ai/bhanujbhalla7-mpstme/cloudscale-ppo-phase1>

You'll see live training metrics, hyperparameters, and model artifacts.

## 7. (Optional) Evaluate the trained policy

After a `single` run, the model is saved to `cloudscale/models/ppo_seed42.zip`:

In [ ]:
%cd /content/CloudScale-AI-Autonomous-FinOps-Orchestrator
!python -m cloudscale.training.eval_policy \
    --model cloudscale/models/ppo_seed42.zip \
    --episodes 5 \
    --out-dir cloudscale/logs/eval

In [ ]:
from IPython.display import Image, display
import os
for f in sorted(os.listdir('cloudscale/logs/eval')):
    if f.endswith('.png'):
        display(Image(filename=f'cloudscale/logs/eval/{f}'))